# Benders Decomposition

**Benders decomposition** {cite:p}`Benders1962` solves block-angular / two-stage problems by splitting the variables into *complicating* (first-stage / master) variables and *recourse* (subproblem) variables. For a fixed master point the recourse subproblem is an independent linear program; its optimal **dual** yields a cut that under-estimates the recourse cost, and its infeasibility yields a cut that excludes the offending master point. Iterating the master/subproblem loop drives the master's lower bound up to meet the incumbent's upper bound.

discopt's `solve_benders` implements **classical Benders** for problems with linear constraints and a linear objective, where every integer variable is first-stage (so the recourse is a continuous LP). Because each cut is a valid *global* under-estimator (LP weak duality holds for the fixed dual point at every master point), the master objective is a **rigorous lower bound** — the solver never reports an unsound bound. Generalized Benders {cite:p}`Geoffrion1972`, which allows convex-NLP recourse, is a planned extension. For a broad survey of the method and its modern variants, see {cite:t}`Rahmaniani2017`.

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

import discopt.modeling as dm

## A two-stage facility problem

Open a facility ($y \in \{0, 1\}$, first stage) and then ship from it ($x_1, x_2 \geq 0$, recourse) to meet a demand of 3 units. Shipping is only possible from an open facility ($x_i \le 5y$).

$$\min_{y, x}\; 2y + x_1 + x_2 \quad\text{s.t.}\quad x_1 + x_2 \ge 3,\; x_1 \le 5y,\; x_2 \le 5y.$$

If $y = 0$ the recourse is **infeasible** (no shipping can meet demand), so Benders adds a *feasibility cut* forcing $y = 1$; then the recourse cost is 3 and the optimum is $2 + 3 = 5$.

In [ ]:
m = dm.Model("facility")
y = m.binary("y")
x1 = m.continuous("x1", lb=0, ub=10)
x2 = m.continuous("x2", lb=0, ub=10)
m.minimize(2 * y + x1 + x2)
m.subject_to(x1 + x2 >= 3)
m.subject_to(x1 <= 5 * y)
m.subject_to(x2 <= 5 * y)

result = m.solve(decomposition="benders")
print("status   :", result.status)
print("objective:", round(result.objective, 4))
print("bound    :", round(result.bound, 4))
print("y =", result.x["y"], " x1 =", result.x["x1"], " x2 =", result.x["x2"])

The reported `bound` equals the objective: the optimality gap is **certified**. This matches the monolithic solve:

In [ ]:
mono = m.solve()
print("monolithic objective:", round(mono.objective, 4))

## Declaring the decomposition structure

By default the **complicating variables are the integer variables** — the canonical Benders split for two-stage mixed-integer programs. For a purely continuous two-stage model, or to override the split, annotate the model directly:

- `m.first_stage(u)` / `m.second_stage(v)` — mark complicating vs recourse variables.
- `m.mark_coupling(constraint)` — mark a linking constraint (used by Lagrangian relaxation).

`discopt.detect_decomposition(model)` resolves these annotations and reports the structure it will use.

In [ ]:
import discopt

lp = dm.Model("continuous_two_stage")
u = lp.continuous("u", lb=0, ub=10)
v = lp.continuous("v", lb=0, ub=10)
lp.first_stage(u)  # u is the complicating variable
lp.minimize(u + 2 * v)
lp.subject_to(u + v >= 4)

print(discopt.detect_decomposition(lp).summary())
print("objective:", round(lp.solve(decomposition="benders").objective, 4))

## When to use Benders

Benders shines when fixing a few complicating variables makes the remaining problem an easy, possibly decomposable, LP — two-stage stochastic programs, network/facility design, and capacity-expansion models. The v1 implementation requires:

- linear constraints and a linear objective, and
- all integer variables in the first stage (continuous recourse).

For nonlinear models, use `m.solve()` (spatial branch & bound) or `m.solve(gdp_method="oa")` (Outer Approximation) {cite:p}`Duran1986` meanwhile; generalized Benders with convex-NLP recourse is on the roadmap.